In [86]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

years = range(2015,2025)
k_vals = [1, 10, 30]

factors_df = pd.read_parquet("../../data/02_clean/fama_french_factors.parquet")
factors_df = factors_df / 100

In [87]:
centralities = ["central", "peripheral"]

portfolios = {}
market_returns_dict = {}

df_mcap = pd.read_csv("../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv")

for year in years:
    for k in k_vals:
        # OOS: Puxa os retornos do ano seguinte (year + 1)
        try:
            oos_ret = pd.read_parquet(f"../../data/02_clean/returns_new_{year+1}.parquet")
        except FileNotFoundError:
            # Se não houver dados para o ano seguinte (ex: último ano da amostra), ele ignora
            continue 
            
        # Retorno de mercado OOS
        market_returns_dict[f"{year}_{k}"] = np.log1p(oos_ret).mean(axis=1)
        
        for centrality in centralities:
            # Pega as empresas da carteira construída no final do ano 'year'
            cols = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date").columns.tolist()
            
            # Filtra apenas os tickers que sobreviveram/existem nos retornos do ano seguinte
            valid_cols = [c for c in cols if c in oos_ret.columns]
            
            # Salva os retornos OOS
            portfolios[f"{centrality}_{year}_{k}"] = oos_ret[valid_cols]


daily_returns = {}

for name, portfolio in portfolios.items():
    year = int(name.split('_')[1])
    k = int(name.split('_')[2])
    
    # Esses log_returns agora são puramente Out-of-Sample!
    log_returns = np.log1p(portfolio)
    R_m = market_returns_dict[f"{year}_{k}"]
    
    # O valor de mercado usado para os pesos DEVE ser o do ano 'year' (In-Sample),
    # pois é a informação disponível na virada do ano para montar a carteira
    mcap_col = f"mcap_{year}"
    if mcap_col not in df_mcap.columns:
        mcap_col = [c for c in df_mcap.columns if "mcap_" in c][-1]
        
    tickers = portfolio.columns.tolist()
    mcaps = df_mcap[df_mcap["Ticker"].isin(tickers)][["Ticker", mcap_col]].copy()
    mcaps[mcap_col] = pd.to_numeric(
        mcaps[mcap_col].astype(str).str.replace(',', '.'), 
        errors='coerce'
    ).fillna(0)

    weights_df = pd.DataFrame({"Ticker": tickers}).merge(mcaps, on="Ticker", how="left").fillna(0)
    w = weights_df[mcap_col].values
    
    if w.sum() == 0:
        w = np.ones(len(w)) / len(w)
    else:
        w = w / w.sum()
        
    # Aplica os pesos construídos em 'year' nos retornos de 'year + 1'
    vw_returns = (log_returns * w).sum(axis=1)

    factors_filtered = factors_df[(factors_df.index >= vw_returns.index[0]) & (factors_df.index <= vw_returns.index[-1])]
    
    # Subtrai a Risk-Free Rate do mesmo período OOS (Lembrando que o factors_df já deve ter sido dividido por 100!)
    daily_returns[name] = vw_returns - factors_filtered["RF"]

In [88]:
import statsmodels.api as sm
import pandas as pd

# 1. Primeiro, vamos juntar (concatenar) os retornos de todos os anos 
# para criar uma série contínua longa para cada combinação de (centrality, k)
portfolio_returns_full = {}

for centrality in centralities:
    for k in k_vals:
        # Pega todas as séries anuais para este k e centralidade que existem no dicionário
        series_list = [daily_returns[f"{centrality}_{year}_{k}"] for year in years if f"{centrality}_{year}_{k}" in daily_returns]
        
        if series_list:
            # Concatena em uma única série temporal
            portfolio_returns_full[f"{centrality}_{k}"] = pd.concat(series_list)

# 2. Agora vamos rodar a regressão de Fama-French (Time-Series Regression)
for name, ret_series in portfolio_returns_full.items():
    
    # Junta os retornos da carteira com os fatores do FF usando as datas (join="inner")
    aligned_data = pd.concat([ret_series.rename("Portfolio_ExRet"), factors_df[["Mkt-RF", "SMB", "HML"]]], axis=1, join="inner")
    
    # Variável dependente (Excesso de retorno da carteira)
    Y = aligned_data["Portfolio_ExRet"]
    
    # Variáveis independentes (Os 3 Fatores)
    X = aligned_data[["Mkt-RF", "SMB", "HML"]]
    
    # Adiciona a constante (Alpha de Jensen)
    X = sm.add_constant(X)
    
    # Roda a regressão OLS
    model = sm.OLS(Y, X).fit()
    
    print(f"=========== Portfólio: {name.upper()} ===========")
    # Mostramos o Alpha e o p-valor (O Alpha é o intercepto 'const')
    # Se p-value < 0.05, significa que a estratégia gerou retorno anormal significativo!
    print(f"Alpha (diário): {model.params['const']:.6f} | P-value: {model.pvalues['const']:.4f}")
    print(f"Mkt-RF Beta:    {model.params['Mkt-RF']:.4f}")
    print(f"SMB Beta:       {model.params['SMB']:.4f}")
    print(f"HML Beta:       {model.params['HML']:.4f}")
    print(f"R-squared:      {model.rsquared:.4f}\n")

    # Se você quiser ver o sumário completo da regressão, você pode descomentar a linha abaixo:
    # print(model.summary())

=========== Portfólio: CENTRAL_1 ===========
Alpha (diário): -0.000151 | P-value: 0.2501
Mkt-RF Beta:    1.2303
SMB Beta:       0.1933
HML Beta:       0.8396
R-squared:      0.8569

=========== Portfólio: CENTRAL_10 ===========
Alpha (diário): -0.000150 | P-value: 0.1173
Mkt-RF Beta:    1.1338
SMB Beta:       0.0618
HML Beta:       0.7922
R-squared:      0.9034

=========== Portfólio: CENTRAL_30 ===========
Alpha (diário): -0.000281 | P-value: 0.0066
Mkt-RF Beta:    1.1580
SMB Beta:       0.0271
HML Beta:       0.8394
R-squared:      0.8941

=========== Portfólio: PERIPHERAL_1 ===========
Alpha (diário): -0.000097 | P-value: 0.4954
Mkt-RF Beta:    0.6233
SMB Beta:       -0.2537
HML Beta:       0.1158
R-squared:      0.4977

=========== Portfólio: PERIPHERAL_10 ===========
Alpha (diário): -0.000130 | P-value: 0.4075
Mkt-RF Beta:    0.6766
SMB Beta:       -0.1751
HML Beta:       0.0341
R-squared:      0.4914

=========== Portfólio: PERIPHERAL_30 ===========
Alpha (diário): -0.000180 | P-

In [90]:
# Adicione este bloco logo abaixo do seu último loop (onde o daily_returns é preenchido)

for year in years:
    for k in k_vals:
        central_name = f"central_{year}_{k}"
        peripheral_name = f"peripheral_{year}_{k}"
        
        # Verifica se ambos os portfólios existem para aquele ano e k
        if central_name in daily_returns and peripheral_name in daily_returns:
            
            ret_central = daily_returns[central_name]
            ret_peripheral = daily_returns[peripheral_name]
            
            # Cria a carteira PMC (Long Peripheral, Short Central)
            pmc_name = f"PMC_{year}_{k}"
            daily_returns[pmc_name] = ret_peripheral - ret_central
            
            # Cria a carteira CMP (Long Central, Short Peripheral)
            cmp_name = f"CMP_{year}_{k}"
            daily_returns[cmp_name] = ret_central - ret_peripheral

print("Carteiras PMC e CMP criadas com sucesso no dicionário daily_returns!")

Carteiras PMC e CMP criadas com sucesso no dicionário daily_returns!


In [91]:
# Atualize a lista no bloco da regressão para incluir as novas carteiras Long-Short
centralities_to_regress = ["PMC", "CMP"]

portfolio_returns_full = {}

for centrality in centralities_to_regress:
    for k in k_vals:
        # Pega todas as séries anuais para este k e centralidade que existem no dicionário
        series_list = [daily_returns[f"{centrality}_{year}_{k}"] for year in years if f"{centrality}_{year}_{k}" in daily_returns]
        
        if series_list:
            # Concatena em uma única série temporal
            portfolio_returns_full[f"{centrality}_{k}"] = pd.concat(series_list)

# 2. Agora vamos rodar a regressão de Fama-French (Time-Series Regression)
for name, ret_series in portfolio_returns_full.items():
    
    # Junta os retornos da carteira com os fatores do FF usando as datas (join="inner")
    aligned_data = pd.concat([ret_series.rename("Portfolio_ExRet"), factors_df[["Mkt-RF", "SMB", "HML"]]], axis=1, join="inner")
    
    # Variável dependente (Excesso de retorno da carteira)
    Y = aligned_data["Portfolio_ExRet"]
    
    # Variáveis independentes (Os 3 Fatores)
    X = aligned_data[["Mkt-RF", "SMB", "HML"]]
    
    # Adiciona a constante (Alpha de Jensen)
    X = sm.add_constant(X)
    
    # Roda a regressão OLS
    model = sm.OLS(Y, X).fit()
    
    print(f"=========== Portfólio: {name.upper()} ===========")
    # Mostramos o Alpha e o p-valor (O Alpha é o intercepto 'const')
    # Se p-value < 0.05, significa que a estratégia gerou retorno anormal significativo!
    print(f"Alpha (diário): {model.params['const']:.6f} | P-value: {model.pvalues['const']:.4f}")
    print(f"Mkt-RF Beta:    {model.params['Mkt-RF']:.4f}")
    print(f"SMB Beta:       {model.params['SMB']:.4f}")
    print(f"HML Beta:       {model.params['HML']:.4f}")
    print(f"R-squared:      {model.rsquared:.4f}\n")

=========== Portfólio: PMC_1 ===========
Alpha (diário): 0.000054 | P-value: 0.8036
Mkt-RF Beta:    -0.6070
SMB Beta:       -0.4470
HML Beta:       -0.7238
R-squared:      0.4705

=========== Portfólio: PMC_10 ===========
Alpha (diário): 0.000020 | P-value: 0.9168
Mkt-RF Beta:    -0.4572
SMB Beta:       -0.2369
HML Beta:       -0.7581
R-squared:      0.4498

=========== Portfólio: PMC_30 ===========
Alpha (diário): 0.000102 | P-value: 0.6651
Mkt-RF Beta:    -0.4655
SMB Beta:       -0.2207
HML Beta:       -0.8157
R-squared:      0.3809

=========== Portfólio: CMP_1 ===========
Alpha (diário): -0.000054 | P-value: 0.8036
Mkt-RF Beta:    0.6070
SMB Beta:       0.4470
HML Beta:       0.7238
R-squared:      0.4705

=========== Portfólio: CMP_10 ===========
Alpha (diário): -0.000020 | P-value: 0.9168
Mkt-RF Beta:    0.4572
SMB Beta:       0.2369
HML Beta:       0.7581
R-squared:      0.4498

=========== Portfólio: CMP_30 ===========
Alpha (diário): -0.000102 | P-value: 0.6651
Mkt-RF Beta:   

In [94]:
df[1][df[1]['node']=='META']

,node,hrm,pozzi,degree,closeness,eig,year,Ticker,Sector,mcap,beta,momentum,log_mcap
